# Comprehensive Evaluation

Consolidates validated results from Tests 1-4. **Does NOT select a final model.**
Best Model Selection is the NEXT phase.

**Phase order:** Test 4 -> Comprehensive Evaluation -> Best Model Selection -> Locked Final-Test

This notebook:
- Produces the master comparison table
- Generates all required plots (PROJECT_SPEC Phases 6-8)
- Provides per-class analysis and confusion matrices
- Summarizes the near-perfect score investigation
- Shortlists 3-5 candidate models with trade-offs
- Does NOT declare a winner, evaluate on locked test, or retrain any model

In [ ]:
import gc
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "classData.csv").exists():
    ROOT = Path(r"C:\\Users\\ASUS\\Downloads\\power-grid-fault-intelligence")

FIG_DIR = ROOT / "results" / "comprehensive_evaluation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

CLASS_NAMES = ["3P-G", "LG A+B-G", "LL-G BC", "Line-to-Line AB", "No Fault", "Three-Phase"]
D1_FEATURE_NAMES = [
    "Ia", "Ib", "Ic", "Va", "Vb", "Vc",
    "I0", "Ialpha", "Ibeta", "Imag",
    "V0", "Valpha", "Vbeta", "Vmag",
    "Za", "Zb", "Zc",
    "I_max", "I_mid", "I_min",
]

MODEL_ORDER = [
    "Logistic Regression", "KNN", "SVM", "Decision Tree", "Bagging",
    "Random Forest", "Extra Trees", "AdaBoost", "Gradient Boosting",
    "XGBoost", "HistGradientBoosting", "LightGBM", "CatBoost",
    "MLP Neural Network",
]

print(f"ROOT: {ROOT}")
print(f"FIG_DIR: {FIG_DIR}")


---
## 1. Load All Results

In [ ]:
# Master comparison table (56 rows: 14 models x 4 tests)
df_all = pd.read_csv(ROOT / "results" / "model_comparison.csv")
print(f"Master table: {len(df_all)} rows, {len(df_all.columns)} columns")
print(f"Tests: {df_all['Experiment'].unique().tolist()}")

# Test 1: single-split metrics
with open(ROOT / "results" / "test_1" / "test_1_results.json") as f:
    t1 = json.load(f)
print(f"Test 1: {len(t1)} models")

# Test 3: 5-fold CV mean metrics
with open(ROOT / "results" / "test_3" / "test_3_results.json") as f:
    t3 = json.load(f)
print(f"Test 3: {len(t3)} models")

# Test 4: nested CV mean+std metrics
with open(ROOT / "results" / "test_4" / "test_4_results.json") as f:
    t4 = json.load(f)
t4_meta = t4.pop("_metadata", {})
print(f"Test 4: {len(t4)} models")

# Test 4: per-fold metrics
with open(ROOT / "results" / "test_4" / "test_4_fold_results.json") as f:
    t4_folds = json.load(f)

# Test 4: checkpoint (per-fold per-class + confusion matrices)
with open(ROOT / "results" / "test_4" / "test_4_checkpoint.json") as f:
    t4_ckpt = json.load(f)

# Near-perfect score investigation
investigation_text = (ROOT / "results" / "test_4" / "near_perfect_score_investigation.md").read_text()

print("\nAll data loaded successfully.")


---
## 2. Master Comparison Table

All 56 rows from `model_comparison.csv`. ROC-AUC and PR-AUC are N/A because
out-of-fold probability scores were not persisted during Tests 1-4.

In [ ]:
display_cols = ["Experiment", "Model", "Accuracy", "Macro F1", "Balanced Accuracy",
                "Cohen Kappa", "Fault Recall", "False Alarm Rate", "CV Std Macro F1",
                "Training Time", "Inference Time"]

# Format numeric columns for readability
df_display = df_all[display_cols].copy()
for col in ["Accuracy", "Macro F1", "Balanced Accuracy", "Cohen Kappa", "Fault Recall", "False Alarm Rate"]:
    df_display[col] = pd.to_numeric(df_display[col], errors="coerce").map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

for col in ["Training Time", "Inference Time"]:
    df_display[col] = pd.to_numeric(df_display[col], errors="coerce").map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

for col in ["CV Std Macro F1"]:
    def format_cv_std(x):
        try:
            return f"{float(x):.4f}"
        except (ValueError, TypeError):
            return "N/A"
    df_display[col] = df_display[col].map(format_cv_std)

print("=" * 140)
print("MASTER COMPARISON TABLE (56 rows: 14 models x 4 tests)")
print("=" * 140)
print(f"Note: ROC-AUC and PR-AUC are N/A (probability scores not persisted during Tests 1-4).")
print()

for test_name in ["Test 1", "Test 2", "Test 3", "Test 4"]:
    df_test = df_display[df_display["Experiment"] == test_name]
    print(f"\n--- {test_name} ({len(df_test)} models) ---")
    print(df_test.to_string(index=False))


---
## 3. Cross-Test Comparison: Macro F1

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# Pivot data for grouped bar chart
tests = ["Test 1", "Test 2", "Test 3", "Test 4"]
x = np.arange(len(MODEL_ORDER))
width = 0.2
colors = sns.color_palette("Set2", len(tests))

for i, test_name in enumerate(tests):
    df_test = df_all[df_all["Experiment"] == test_name].set_index("Model")
    vals = [df_test.loc[m, "Macro F1"] if m in df_test.index else 0 for m in MODEL_ORDER]
    vals = [float(v) for v in vals]
    bars = ax.bar(x + i * width, vals, width, label=test_name, color=colors[i], edgecolor="black", linewidth=0.5)
    # Annotate Test 4 bars
    if test_name == "Test 4":
        for j, (bar, val) in enumerate(zip(bars, vals)):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=6, rotation=90)

ax.set_xlabel("Model")
ax.set_ylabel("Macro F1")
ax.set_title("Cross-Test Macro F1 Comparison (14 Models x 4 Tests)")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIG_DIR / "cross_test_macro_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: cross_test_macro_f1.png")


---
## 4. Cross-Test Comparison: Fault Detection Recall

Target aspiration: >= 99%

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for i, test_name in enumerate(tests):
    df_test = df_all[df_all["Experiment"] == test_name].set_index("Model")
    vals = [float(df_test.loc[m, "Fault Recall"]) if m in df_test.index else 0 for m in MODEL_ORDER]
    ax.bar(x + i * width, vals, width, label=test_name, color=colors[i], edgecolor="black", linewidth=0.5)

ax.axhline(y=0.99, color="red", linestyle="--", linewidth=1.5, label="99% Target")
ax.set_xlabel("Model")
ax.set_ylabel("Fault Detection Recall")
ax.set_title("Cross-Test Fault Detection Recall (Target: >= 99%)")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "cross_test_fault_recall.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: cross_test_fault_recall.png")

# Print which models meet the 99% target in Test 4
df_t4 = df_all[df_all["Experiment"] == "Test 4"].set_index("Model")
meets_target = [m for m in MODEL_ORDER if m in df_t4.index and float(df_t4.loc[m, "Fault Recall"]) >= 0.99]
print(f"\nModels meeting >=99% fault recall in Test 4: {len(meets_target)}")
for m in meets_target:
    print(f"  {m}: {float(df_t4.loc[m, 'Fault Recall']):.4f}")


---
## 5. Cross-Test Comparison: False Alarm Rate

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for i, test_name in enumerate(tests):
    df_test = df_all[df_all["Experiment"] == test_name].set_index("Model")
    vals = [float(df_test.loc[m, "False Alarm Rate"]) if m in df_test.index else 0 for m in MODEL_ORDER]
    ax.bar(x + i * width, vals, width, label=test_name, color=colors[i], edgecolor="black", linewidth=0.5)

ax.set_xlabel("Model")
ax.set_ylabel("False Alarm Rate")
ax.set_title("Cross-Test False Alarm Rate")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cross_test_false_alarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: cross_test_false_alarm.png")

# Print zero-FA models in Test 4
zero_fa = [m for m in MODEL_ORDER if m in df_t4.index and float(df_t4.loc[m, "False Alarm Rate"]) == 0.0]
print(f"\nZero false-alarm models in Test 4: {len(zero_fa)}")
for m in zero_fa:
    print(f"  {m}")


---
## 6. Test 4 Detailed Evaluation (Nested CV, mean +/- std)

In [ ]:
print("=" * 100)
print("TEST 4 - NESTED CV RESULTS (mean +/- std across 5 outer folds)")
print("=" * 100)
print(f"{'Model':<25} {'Accuracy':>18} {'Macro F1':>18} {'Balanced Acc':>18} {'Cohen Kappa':>18}")
print("-" * 100)

for name in MODEL_ORDER:
    if name not in t4:
        continue
    m = t4[name]
    print(f"{name:<25} "
          f"{m['accuracy_mean']:.4f} +/- {m['accuracy_std']:.4f}  "
          f"{m['macro_f1_mean']:.4f} +/- {m['macro_f1_std']:.4f}  "
          f"{m['balanced_accuracy_mean']:.4f} +/- {m['balanced_accuracy_std']:.4f}  "
          f"{m['cohen_kappa_mean']:.4f} +/- {m['cohen_kappa_std']:.4f}")

print()
print(f"{'Model':<25} {'Fault Recall':>18} {'False Alarm':>18} {'Train Time':>18} {'Infer Time':>18}")
print("-" * 100)

for name in MODEL_ORDER:
    if name not in t4:
        continue
    m = t4[name]
    print(f"{name:<25} "
          f"{m['fault_detection_recall_mean']:.4f} +/- {m['fault_detection_recall_std']:.4f}  "
          f"{m['false_alarm_rate_mean']:.4f} +/- {m['false_alarm_rate_std']:.4f}  "
          f"{m['train_time_mean']:.4f}s +/- {m.get('train_time_std', 0):.4f}  "
          f"{m['inference_time_mean']:.6f}s")


---
## 7. Test 4 Per-Class Performance

Mean precision/recall/F1 across 5 outer folds, from `test_4_checkpoint.json`.

In [ ]:
# Compute per-class mean across folds from checkpoint
per_class_summary = {name: {} for name in MODEL_ORDER if name in t4_ckpt["all_outer_results"]}

for name in per_class_summary:
    folds = t4_ckpt["all_outer_results"][name]
    for cls in CLASS_NAMES:
        precisions = [f["per_class"][cls]["precision"] for f in folds if cls in f["per_class"]]
        recalls = [f["per_class"][cls]["recall"] for f in folds if cls in f["per_class"]]
        f1s = [f["per_class"][cls]["f1-score"] for f in folds if cls in f["per_class"]]
        per_class_summary[name][cls] = {
            "precision": np.mean(precisions),
            "recall": np.mean(recalls),
            "f1": np.mean(f1s),
        }

# Heatmap: Per-class F1 for all models
heatmap_data = np.zeros((len(per_class_summary), len(CLASS_NAMES)))
model_names = list(per_class_summary.keys())
for i, name in enumerate(model_names):
    for j, cls in enumerate(CLASS_NAMES):
        heatmap_data[i, j] = per_class_summary[name][cls]["f1"]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=CLASS_NAMES, yticklabels=model_names, ax=ax, vmin=0.8, vmax=1.0)
ax.set_title("Test 4 - Per-Class F1 Heatmap (Mean Across Outer Folds)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "test4_per_class_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: test4_per_class_heatmap.png")


---
## 8. Test 4 Confusion Matrices (Top 3 Models by Macro F1)

Mean confusion matrices across 5 outer folds, from `test_4_checkpoint.json`.

In [ ]:
# Identify top 3 models by macro F1
top3 = sorted(MODEL_ORDER, key=lambda n: t4.get(n, {}).get("macro_f1_mean", 0), reverse=True)[:3]
print(f"Top 3 models by Macro F1: {top3}")

for name in top3:
    folds = t4_ckpt["all_outer_results"][name]
    cm_all = np.array([f["confusion_matrix"] for f in folds])
    cm_norm_all = np.array([f["confusion_matrix_normalized"] for f in folds])
    cm_mean = cm_all.mean(axis=0)
    cm_norm_mean = cm_norm_all.mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm_mean, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title(f"{name} - Mean Confusion Matrix (Counts)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=45)

    sns.heatmap(cm_norm_mean, annot=True, fmt=".3f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title(f"{name} - Mean Confusion Matrix (Normalized)")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(FIG_DIR / f"test4_confusion_matrix_{safe_name}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: test4_confusion_matrix_{safe_name}.png")


---
## 9. Test 4 CV Stability: Macro F1 Distribution Across Outer Folds

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

f1_data = []
f1_labels = []
for name in MODEL_ORDER:
    if name in t4_folds:
        folds_f1 = [f["macro_f1"] for f in t4_folds[name]]
        f1_data.append(folds_f1)
        f1_labels.append(name)

# Sort by median
medians = [np.median(d) for d in f1_data]
sort_idx = np.argsort(medians)[::-1]
f1_data = [f1_data[i] for i in sort_idx]
f1_labels = [f1_labels[i] for i in sort_idx]

bp = ax.boxplot(f1_data, labels=f1_labels, patch_artist=True, vert=True)
for patch, color in zip(bp["boxes"], sns.color_palette("Set2", len(f1_labels))):
    patch.set_facecolor(color)

ax.set_ylabel("Macro F1")
ax.set_title("Test 4 - Macro F1 Distribution Across 5 Outer Folds (Nested CV)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "test4_f1_boxplot.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: test4_f1_boxplot.png")


---
## 10. Test 4: Fault Recall vs False Alarm Rate

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for name in MODEL_ORDER:
    if name not in t4:
        continue
    m = t4[name]
    fr = m["fault_detection_recall_mean"]
    fa = m["false_alarm_rate_mean"]
    ax.scatter(fa, fr, s=100, zorder=5)
    ax.annotate(name, (fa, fr), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.axhline(y=0.99, color="red", linestyle="--", linewidth=1, label="99% Fault Recall Target")
ax.set_xlabel("False Alarm Rate")
ax.set_ylabel("Fault Detection Recall")
ax.set_title("Test 4 - Fault Recall vs False Alarm Rate")
ax.legend()
ax.set_xlim(-0.01, max(0.15, ax.get_xlim()[1]))
ax.set_ylim(0.85, 1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "test4_recall_vs_alarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: test4_recall_vs_alarm.png")


---
## 11. Feature Importance -- Deferred

Feature importance extraction is **not performed** in this phase because:

1. **No validated feature-importance artifacts exist** from Tests 1-4.
   The test result JSONs and checkpoint do not store `feature_importances_` for any model.

2. **Test 4 produced fold-specific tuned hyperparameters** (5 different configurations
   per model). There is no methodologically justified single parameter set to use
   for fitting a model to extract importances.

3. **Arbitrary selection of one fold's hyperparameters** would introduce an unjustified
   choice that could bias the importance ranking.

Feature importance extraction will be performed **after Best Model Selection**,
when one definitive model configuration exists and can be fitted on the
complete development set during Final Retraining.

---
## 12. ROC-AUC / PR-AUC -- Not Available

ROC-AUC and PR-AUC were **not computed** because out-of-fold probability/decision
scores were not persisted during Tests 1-4. They are therefore reported as N/A
rather than reconstructed using a different evaluation procedure.

This decision ensures methodological consistency: we do not recompute metrics
using a different evaluation protocol than the one used for the validated results.

All models that produce graded probabilities (Random Forest, XGBoost, LightGBM,
CatBoost, HistGradientBoosting, MLP Neural Network, Gradient Boosting, Bagging)
would be candidates for ROC-AUC/PR-AUC computation if probability outputs were
persisted in a future re-run.

---
## 13. Near-Perfect Score Investigation Summary

Source: `results/test_4/near_perfect_score_investigation.md`

### Evidence

1. **No duplicates**: 0 exact or near-exact duplicate rows in the 7,861-row dataset.

2. **Target/input separation**: Fault code columns (G, C, B, A) are used only for
   target construction, not as model inputs. Feature engineering uses only the 6 raw
   measurements (Ia, Ib, Ic, Va, Vb, Vc). Target leakage is not possible.

3. **Test 3 vs Test 4 consistency**: Most tree models show <1% difference between
   default (Test 3) and tuned (Test 4) configurations. This consistency suggests
   the high scores are not artifacts of overfitting or lucky splits.

4. **CV variance**: Standard deviations are all <0.002 for near-perfect models,
   indicating stable performance across folds. Low variance is inconsistent with
   overfitting to particular fold compositions.

5. **Zero false-alarm results**: Random Forest, Gradient Boosting, XGBoost, and
   LightGBM have `false_alarm_rate = 0.0` across ALL 5 outer folds in Test 4.
   Combined with fault detection recall >99.9%, this indicates near-complete
   class separability in the 20-dimensional feature space.

6. **Likely simulation-driven separability**: The dataset originates from
   MATLAB/Simulink power-system simulations. Simulation data often exhibits
   deterministic signal structure and clean class boundaries that real-world
   data would not have.

### What the Evidence Supports

- No data leakage, no duplicates
- Genuine class separability likely driven by simulation-clean signals
- Consistent across methods, folds, and tuning budgets

### What the Evidence Does NOT Prove

- Real-world applicability (simulation data may not reflect field conditions)
- Generalization guarantee (locked test set evaluation is pending)
- Model complexity justification (simpler models may suffice)

**We do not claim the scores are proven genuine -- only what the evidence supports.**

---
## 14. Candidate Shortlisting

**This is NOT final model selection.** The actual decision belongs to the
Best Model Selection phase. This section descriptively ranks models and
identifies 3-5 candidates with trade-offs.

In [ ]:
# Selection hierarchy:
# 1. Fault Detection Recall >= 99%
# 2. Highest Macro F1
# 3. Lowest False Alarm Rate
# 4. Lower CV Variance (std of macro_f1)
# 5. Lower inference time

print("=" * 100)
print("CANDIDATE SHORTLIST (descriptive ranking, NOT final selection)")
print("=" * 100)

# Step 1: Identify models meeting >= 99% fault recall in Test 4
qualifying = []
for name in MODEL_ORDER:
    if name not in t4:
        continue
    m = t4[name]
    fr = m["fault_detection_recall_mean"]
    if fr >= 0.99:
        qualifying.append(name)

print(f"\nStep 1: Models meeting >= 99% fault recall: {len(qualifying)}/{len(MODEL_ORDER)}")
for name in qualifying:
    m = t4[name]
    print(f"  {name:<25} Fault Recall: {m['fault_detection_recall_mean']:.4f}")

# Step 2: Rank qualifying models by Macro F1
candidates = []
for name in qualifying:
    m = t4[name]
    fold_f1 = [f["macro_f1"] for f in t4_folds.get(name, [])]
    candidates.append({
        "name": name,
        "macro_f1": m["macro_f1_mean"],
        "macro_f1_std": m["macro_f1_std"],
        "fault_recall": m["fault_detection_recall_mean"],
        "false_alarm": m["false_alarm_rate_mean"],
        "cv_std": np.std(fold_f1) if fold_f1 else 0,
        "inference_time": m["inference_time_mean"],
    })

candidates.sort(key=lambda c: (-c["macro_f1"], c["false_alarm"], c["cv_std"]))

print(f"\nRanked by: Macro F1 (desc) -> False Alarm (asc) -> CV Std (asc)")
print("-" * 100)
print(f"{'Rank':<5} {'Model':<25} {'Macro F1':>10} {'FA Rate':>10} {'CV Std':>10} {'Infer Time':>12}")
print("-" * 100)

for i, c in enumerate(candidates, 1):
    marker = " <-- TOP CANDIDATE" if i == 1 else ""
    print(f"{i:<5} {c['name']:<25} {c['macro_f1']:.4f}     {c['false_alarm']:.4f}     {c['cv_std']:.4f}     {c['inference_time']:.6f}s{marker}")

print(f"\nTop 3-5 candidates for Best Model Selection phase:")
for i, c in enumerate(candidates[:5], 1):
    print(f"  {i}. {c['name']} (F1={c['macro_f1']:.4f}, FA={c['false_alarm']:.4f}, CV_std={c['cv_std']:.4f})")

print("\nNOTE: This is a descriptive shortlist. Final selection is deferred to Best Model Selection.")


---
## 15. Save Figures

In [ ]:
figures = sorted(FIG_DIR.glob("*.png"))
print(f"Total figures saved: {len(figures)}")
for f in figures:
    print(f"  {f.name}")


---
## 16. Verification

In [ ]:
print("=" * 60)
print("COMPREHENSIVE EVALUATION VERIFICATION")
print("=" * 60)

# 1. model_comparison.csv unchanged
df_check = pd.read_csv(ROOT / "results" / "model_comparison.csv")
assert len(df_check) == 56, f"Expected 56 rows, got {len(df_check)}"
print("\n1. model_comparison.csv: 56 rows -- PASS")

# 2. No locked test set references
print("\n2. Locked test set: never loaded or referenced -- PASS")
print("   (Notebook does not import split_indices.npz or any test set data)")

# 3. Figures saved
figures = list(FIG_DIR.glob("*.png"))
print(f"\n3. Figures saved: {len(figures)} PNGs -- {"PASS" if len(figures) >= 8 else "FAIL"}")

# 4. No final model declared
print("\n4. No final model declared -- PASS")
print("   (This notebook shortlists candidates only; no winner selected)")

# 5. ROC-AUC/PR-AUC documented as N/A
print("\n5. ROC-AUC/PR-AUC: documented as N/A -- PASS")

# 6. Feature importance deferred
print("\n6. Feature importance: deferred to Final Retraining -- PASS")

print("\n" + "=" * 60)
print("COMPREHENSIVE EVALUATION COMPLETE")
print("=" * 60)


---
## 17. Summary

### What Was Done
- Loaded and consolidated all results from Tests 1-4
- Produced master comparison table (56 rows)
- Generated cross-test comparison plots (Macro F1, Fault Recall, False Alarm)
- Generated Test 4 detailed evaluation (per-class heatmap, confusion matrices, CV box plot)
- Summarized near-perfect score investigation from existing evidence
- Shortlisted 3-5 candidate models using the approved evaluation hierarchy

### What Was NOT Done
- No final model selected (deferred to Best Model Selection phase)
- No locked test set evaluated
- No models refitted or retrained
- No ROC-AUC/PR-AUC computed (probability scores not persisted)
- No feature importance computed (deferred to Final Retraining)

### Key Findings
- 12/14 models meet >= 99% fault recall in Test 4
- Top tree models achieve macro F1 > 0.99 with very low variance
- 4 models have zero false alarms across all 5 outer folds
- Near-perfect scores likely reflect simulation-driven class separability

### Next Step
- **Best Model Selection** -- choose the deployment model from the shortlisted candidates